In [1]:
import duckdb
import pandas as pd
from pathlib import Path

PROJECT = Path(
    "/Users/anjangangisetty/Desktop/pricing-elasticity-project"
)

PYTHON_RESULTS = PROJECT / "outputs" / "python_results"
POWERBI = PROJECT / "outputs" / "powerbi"

POWERBI.mkdir(parents=True, exist_ok=True)
(PROJECT / "sql").mkdir(parents=True, exist_ok=True)

con = duckdb.connect(str(PROJECT / "data" / "pricing.duckdb"))

print("Database connected.")

Database connected.


In [2]:
sales_path = (
    PROJECT / "data" / "processed" / "cereal_analysis.parquet"
).as_posix()

con.execute(f"""
CREATE OR REPLACE TABLE sales AS
SELECT
    CAST(STORE AS INTEGER) AS store_id,
    CAST(UPC AS VARCHAR) AS upc,
    CAST(WEEK AS INTEGER) AS week,
    CAST(MOVE AS BIGINT) AS units,
    CAST(UNIT_PRICE AS DOUBLE) AS unit_price,
    CAST(REVENUE AS DOUBLE) AS revenue,
    TRIM(DESCRIP) AS product,
    TRIM(SIZE) AS size,
    CASE
        WHEN SALE IS NULL OR TRIM(SALE) = '' THEN 0
        ELSE 1
    END AS promo_recorded
FROM read_parquet('{sales_path}');
""")

con.sql("SELECT * FROM sales LIMIT 5").df()

,store_id,upc,week,units,unit_price,revenue,product,size,promo_recorded
0,2,1313000002,1,19,1.69,32.11,NAB SHREDDED WHEAT,10 OZ,0
1,2,1313000002,2,21,1.69,35.49,NAB SHREDDED WHEAT,10 OZ,0
2,2,1313000002,3,17,1.69,28.73,NAB SHREDDED WHEAT,10 OZ,0
3,2,1313000002,4,25,1.69,42.25,NAB SHREDDED WHEAT,10 OZ,0
4,2,1313000002,5,27,1.69,45.63,NAB SHREDDED WHEAT,10 OZ,0


In [3]:
for table_name in [
    "product_elasticities",
    "pricing_scenarios",
    "pricing_grid"
]:
    file_path = (
        PYTHON_RESULTS / f"{table_name}.csv"
    ).as_posix()

    con.execute(f"""
        CREATE OR REPLACE TABLE {table_name} AS
        SELECT *
        FROM read_csv(
            '{file_path}',
            header = true,
            types = {{'upc': 'VARCHAR'}}
        );
    """)

con.sql("SHOW TABLES").df()

,name
0,pricing_grid
1,pricing_scenarios
2,product_elasticities
3,sales


In [4]:
con.sql("""
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT upc) AS products,
    COUNT(DISTINCT store_id) AS stores,
    MIN(week) AS first_week,
    MAX(week) AS last_week,
    SUM(units) AS total_units,
    SUM(revenue) AS total_revenue
FROM sales;
""").df()

,row_count,products,stores,first_week,last_week,total_units,total_revenue
0,4706287,485,93,1,399,90708109.0,2.619889e+08


In [5]:
con.sql("""
SELECT
    store_id,
    upc,
    week,
    COUNT(*) AS row_count
FROM sales
GROUP BY store_id, upc, week
HAVING COUNT(*) > 1
LIMIT 20;
""").df()

,store_id,upc,week,row_count


In [6]:
con.sql("""
SELECT
    upc
FROM sales
GROUP BY upc
HAVING
    COUNT(DISTINCT product) > 1
    OR COUNT(DISTINCT size) > 1;
""").df()

,upc


In [7]:
reporting_sql = """
CREATE OR REPLACE VIEW dim_product AS
SELECT DISTINCT
    upc,
    product,
    size,
    product || ' | ' || size || ' | ' || upc AS product_label
FROM sales;

CREATE OR REPLACE VIEW weekly_sales AS
SELECT
    upc,
    week,
    promo_recorded,
    SUM(units) AS units,
    SUM(revenue) AS revenue,
    COUNT(*) AS source_observations
FROM sales
GROUP BY
    upc,
    week,
    promo_recorded;
"""

con.execute(reporting_sql)

(PROJECT / "sql" / "01_reporting_views.sql").write_text(
    reporting_sql
)

con.sql("SELECT * FROM weekly_sales LIMIT 10").df()

,upc,week,promo_recorded,units,revenue,source_observations
0,1600065620,347,0,832.0,2762.24,79
1,1600065620,355,0,1030.0,3432.54,76
2,1600065620,363,0,929.0,3094.83,76
3,1600065620,365,0,990.0,3291.90,78
4,1600065620,369,0,894.0,2971.18,78
5,1600065620,379,0,876.0,2916.58,80
6,1600065620,382,0,783.0,2606.49,80
7,1600065620,385,0,856.0,2845.12,80
8,1600065620,386,0,959.0,3189.17,80
9,1600065620,197,0,640.0,1820.24,70


In [8]:
con.sql("""
SELECT
    p.product_label,
    SUM(w.units) AS total_units,
    ROUND(SUM(w.revenue), 2) AS total_revenue,
    ROUND(
        SUM(w.revenue) / NULLIF(SUM(w.units), 0),
        2
    ) AS average_selling_price
FROM weekly_sales AS w
JOIN dim_product AS p
    ON w.upc = p.upc
GROUP BY p.product_label
ORDER BY total_revenue DESC
LIMIT 10;
""").df()

,product_label,total_units,total_revenue,average_selling_price
0,CHEERIOS | 15 OZ | 1600066610,2011819.0,5715980.70,2.84
1,HONEY NUT CHEERIOS | 14 OZ | 1600066590,1512370.0,4154655.73,2.75
2,KELLOGG'S FROSTED FL | 20 OZ | 3800001520,1322703.0,3931466.34,2.97
3,KELLOGGS FRUIT LOOPS | 15 OZ | 3800001720,1208611.0,3589030.75,2.97
4,KELLOGGS CORN FLAKES | 18 OZ | 3800000120,1830189.0,3488292.49,1.91
5,CHEERIOS | 10 OZ | 1600066510,1560241.0,3464034.91,2.22
6,KELLOGG'S FROSTED FL | 15 OZ | 3800001510,1287272.0,3326669.72,2.58
7,KELLOGGS SPECIAL K | 18 OZ | 3800001621,814409.0,3143528.92,3.86
8,KELLOGGS CORN POPS | 15 OZ | 3800001011,1010399.0,3132156.62,3.10
9,KELLOGGS SPECIAL K | 12 OZ | 3800001611,1077255.0,3120524.23,2.90


In [9]:
con.sql("""
SELECT
    (SELECT SUM(units) FROM sales)
        - (SELECT SUM(units) FROM weekly_sales)
        AS units_difference,

    ROUND(
        (SELECT SUM(revenue) FROM sales)
        - (SELECT SUM(revenue) FROM weekly_sales),
        2
    ) AS revenue_difference;
""").df()

,units_difference,revenue_difference
0,0.0,-0.0


In [10]:
tables = [
    "dim_product",
    "weekly_sales",
    "product_elasticities",
    "pricing_scenarios",
    "pricing_grid"
]

for table_name in tables:
    result = con.sql(f"SELECT * FROM {table_name}").df()
    destination = POWERBI / f"{table_name}.csv"
    result.to_csv(destination, index=False)

    print(f"{table_name}: {len(result):,} rows exported")

con.close()

dim_product: 485 rows exported
weekly_sales: 77,436 rows exported
product_elasticities: 10 rows exported
pricing_scenarios: 5 rows exported
pricing_grid: 105 rows exported
